# FitPredict
## Machine Learning-Based Calorie Expenditure Prediction System

### Objective
To develop a supervised machine learning regression model
that predicts calorie expenditure using exercise-related
and physiological measurements.



In [1]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]
Python executable: C:\Users\user\AppData\Local\Programs\Python\Python311\python.exe


In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt 
import seaborn as sns
from datasets import load_dataset

In [2]:
exercise=pd.read_csv( "https://huggingface.co/datasets/mnemoraorg/calorie-burnt-15k/resolve/main/raw_exercise.csv")
calories=pd.read_csv( "https://huggingface.co/datasets/mnemoraorg/calorie-burnt-15k/resolve/main/raw_calories.csv")


### Inspecting the two datasets

In [3]:
display(exercise.head())

,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7
3,16180408,female,34,179.0,71.0,13.0,100.0,40.5
4,17771927,female,27,154.0,58.0,10.0,81.0,39.8


In [4]:
display(calories.head())

,User_ID,Calories
0,14733363,231.0
1,14861698,66.0
2,11179863,26.0
3,16180408,71.0
4,17771927,35.0


In [5]:
print("Exercise shape:",exercise.shape)

Exercise shape: (15000, 8)


In [6]:
print("Calories shape:",calories.shape)

Calories shape: (15000, 2)


### Merging the dataset using User_ID

In [9]:
df=pd.merge(exercise,calories,on="User_ID",how="inner",validate="one_to_one")

### Inspecting

In [10]:
display(df.head())

,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,16180408,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,17771927,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [11]:
print("Merged dataset shape:",df.shape)

Merged dataset shape: (15000, 9)


In [12]:
print(df.columns)

Index(['User_ID', 'Gender', 'Age', 'Height', 'Weight', 'Duration',
       'Heart_Rate', 'Body_Temp', 'Calories'],
      dtype='str')


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   User_ID     15000 non-null  int64  
 1   Gender      15000 non-null  str    
 2   Age         15000 non-null  int64  
 3   Height      15000 non-null  float64
 4   Weight      15000 non-null  float64
 5   Duration    15000 non-null  float64
 6   Heart_Rate  15000 non-null  float64
 7   Body_Temp   15000 non-null  float64
 8   Calories    15000 non-null  float64
dtypes: float64(6), int64(2), str(1)
memory usage: 1.1 MB


In [14]:
print(df.isnull().sum())

User_ID       0
Gender        0
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
Calories      0
dtype: int64


In [15]:
print(df.duplicated().sum())

0


In [16]:
print("Duplicate User IDs:",df["User_ID"].duplicated().sum())

Duplicate User IDs: 0


In [17]:
print(df["Gender"].unique())

<ArrowStringArray>
['male', 'female']
Length: 2, dtype: str


In [18]:
print(df["Gender"].value_counts())

Gender
female    7553
male      7447
Name: count, dtype: int64


In [19]:
df.describe()

,User_ID,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
count,1.500000e+04,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,1.497736e+07,42.789800,174.465133,74.966867,15.530600,95.518533,40.025453,89.539533
std,2.872851e+06,16.980264,14.258114,15.035657,8.319203,9.583328,0.779230,62.456978
min,1.000116e+07,20.000000,123.000000,36.000000,1.000000,67.000000,37.100000,1.000000
25%,1.247419e+07,28.000000,164.000000,63.000000,8.000000,88.000000,39.600000,35.000000
50%,1.499728e+07,39.000000,175.000000,74.000000,16.000000,96.000000,40.200000,79.000000
75%,1.744928e+07,56.000000,185.000000,87.000000,23.000000,103.000000,40.600000,138.000000
max,1.999965e+07,79.000000,222.000000,132.000000,30.000000,128.000000,41.500000,314.000000


In [22]:
columns=[ 'Age', 'Height', 'Weight', 'Duration',
       'Heart_Rate', 'Body_Temp', 'Calories']

for column in columns:
    print(column,"Minimum:",df[column].min(),"Maximum:",df[column].max())

Age Minimum: 20 Maximum: 79
Height Minimum: 123.0 Maximum: 222.0
Weight Minimum: 36.0 Maximum: 132.0
Duration Minimum: 1.0 Maximum: 30.0
Heart_Rate Minimum: 67.0 Maximum: 128.0
Body_Temp Minimum: 37.1 Maximum: 41.5
Calories Minimum: 1.0 Maximum: 314.0


### Identifying potential outliers

In [23]:
for column in columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_limit) |
        (df[column] > upper_limit)
    ]

    print(column, "Potential outliers:", len(outliers))

Age Potential outliers: 0
Height Potential outliers: 14
Weight Potential outliers: 6
Duration Potential outliers: 0
Heart_Rate Potential outliers: 1
Body_Temp Potential outliers: 369
Calories Potential outliers: 4
